<a href="https://colab.research.google.com/github/jithender-4914/UrbanFlowAI/blob/main/nndl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [ ]:
crime_hist = pd.read_csv("/content/drive/MyDrive/nndl/StationFootfall_Merged_2019-2023.csv")
crime_recent = pd.read_csv("/content/drive/MyDrive/nndl/historical_crime.csv")
housing = pd.read_csv("/content/drive/MyDrive/nndl/housing-value.csv")
gdf = gpd.read_file("/content/drive/MyDrive/nndl/lsoa_shapefile/lsoa/01_LondonLSOA.shp")

In [ ]:
print(gdf.columns)
print(len(gdf))
print(crime_recent.columns)
print(crime_recent.head())

Index(['code', 'name', 'label', 'Area', 'Half densi', 'Area2', 'geometry'], dtype='object')
4719
Index(['LSOA Code', 'LSOA Name', 'Borough', 'Major Category', 'Minor Category',
       '201903', '201904', '201905', '201906', '201907',
       ...
       '202206', '202207', '202208', '202209', '202210', '202211', '202212',
       '202301', '202302', '202303'],
      dtype='object', length=161)
   LSOA Code                  LSOA Name    Borough             Major Category  \
0  E01000006  Barking and Dagenham 016A  E09000002  ARSON AND CRIMINAL DAMAGE   
1  E01000006  Barking and Dagenham 016A  E09000002  ARSON AND CRIMINAL DAMAGE   
2  E01000006  Barking and Dagenham 016A  E09000002                   BURGLARY   
3  E01000006  Barking and Dagenham 016A  E09000002                   BURGLARY   
4  E01000006  Barking and Dagenham 016A  E09000002              DRUG OFFENCES   

                    Minor Category  201903  201904  201905  201906  201907  \
0                            ARSON       

In [ ]:
gdf[['code','name']].head()

,code,name
0,E01000037,Barking and Dagenham 003B
1,E01033729,Greenwich 030E
2,E01000038,Barking and Dagenham 003C
3,E01033730,Greenwich 035D
4,E01000039,Barking and Dagenham 003D


In [ ]:
crime_grouped = crime_recent.groupby("LSOA Code").sum(numeric_only=True)

print("Crime grouped shape:", crime_grouped.shape)
print("First 5 LSOA codes:")
print(crime_grouped.index[:5])

Crime grouped shape: (4988, 156)
First 5 LSOA codes:
Index(['E01000006', 'E01000007', 'E01000008', 'E01000009', 'E01000011'], dtype='object', name='LSOA Code')


In [ ]:
# Get LSOA codes from shapefile
lsoa_geo = set(gdf["code"])

# Get LSOA codes from crime data
lsoa_crime = set(crime_grouped.index)

# Find intersection
common_lsoa = lsoa_geo.intersection(lsoa_crime)

print("LSOAs in shapefile:", len(lsoa_geo))
print("LSOAs in crime data:", len(lsoa_crime))
print("Common LSOAs:", len(common_lsoa))

LSOAs in shapefile: 4719
LSOAs in crime data: 4988
Common LSOAs: 4541


In [ ]:
crime_london = crime_grouped.loc[list(common_lsoa)]
print("Filtered crime shape:", crime_london.shape)

Filtered crime shape: (4541, 156)


In [ ]:
missing_lsoa = set(gdf["code"]) - set(crime_grouped.index)

print("Missing LSOAs:", len(missing_lsoa))
print(list(missing_lsoa)[:10])

Missing LSOAs: 178
['E01033492', 'E01004142', 'E01004246', 'E01002895', 'E01032720', 'E01000010', 'E01002532', 'E01004318', 'E01004739', 'E01001117']


In [ ]:
month_cols = [col for col in crime_recent.columns if col.isdigit()]

print("Number of month columns:", len(month_cols))
print(month_cols[:10])
crime_grouped = (
    crime_recent
    .groupby("LSOA Code")[month_cols]
    .sum()
)

print("Clean grouped shape:", crime_grouped.shape)
crime_london = crime_grouped.reindex(gdf["code"]).fillna(0)

print("Final aligned shape:", crime_london.shape)
print("Total zero rows:",
      (crime_london.sum(axis=1) == 0).sum())

Number of month columns: 156
['201903', '201904', '201905', '201906', '201907', '201908', '201909', '201910', '201911', '201912']
Clean grouped shape: (4988, 156)
Final aligned shape: (4719, 156)
Total zero rows: 178


In [ ]:
print(month_cols[-10:])

['202206', '202207', '202208', '202209', '202210', '202211', '202212', '202301', '202302', '202303']


In [ ]:
print(len(month_cols))
print(sorted(month_cols)[:5])
print(sorted(month_cols)[-5:])

156
['201004', '201005', '201006', '201007', '201008']
['202211', '202212', '202301', '202302', '202303']


In [ ]:
# Sort columns chronologically
crime_matrix = crime_london[sorted(month_cols)]

# Use last 60 months
crime_matrix = crime_matrix.iloc[:, -60:]

print(crime_matrix.shape)

(4719, 60)


In [ ]:
crime_ts = crime_matrix.T
print(crime_ts.shape)

(60, 4719)


In [ ]:
print(crime_ts.iloc[:3, :5])

code    E01000037  E01033729  E01000038  E01033730  E01000039
201804        7.0       13.0        6.0        0.0        4.0
201805        3.0       10.0        6.0        0.0        4.0
201806        4.0        9.0        5.0        0.0        8.0


In [ ]:
import numpy as np

# Compute mean and std per LSOA (per column)
mean_per_node = crime_ts.mean(axis=0)
std_per_node = crime_ts.std(axis=0)

# Avoid division by zero
std_per_node[std_per_node == 0] = 1

# Normalize
crime_ts_norm = (crime_ts - mean_per_node) / std_per_node

print(crime_ts_norm.iloc[:3, :5])

code    E01000037  E01033729  E01000038  E01033730  E01000039
201804  -0.572387  -0.448642   0.203125        0.0  -0.737131
201805  -1.613092  -1.009444   0.203125        0.0  -0.737131
201806  -1.352916  -1.196378  -0.145090        0.0   0.517560


In [ ]:
T_window = 12

X_list = []
Y_list = []

values = crime_ts_norm.values  # convert to numpy

for i in range(len(values) - T_window):
    X_list.append(values[i:i+T_window])
    Y_list.append(values[i+T_window])

X = np.array(X_list)
Y = np.array(Y_list)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (48, 12, 4719)
Y shape: (48, 4719)


In [ ]:
import numpy as np
import geopandas as gpd

# Ensure same order as crime_london
gdf = gdf.set_index("code")
gdf = gdf.loc[crime_london.index]

# Reset index for spatial operations
gdf = gdf.reset_index()

# Build adjacency using spatial intersection
neighbors = gpd.sjoin(gdf, gdf, predicate="touches")

# Remove self-joins
neighbors = neighbors[neighbors["code_left"] != neighbors["code_right"]]

# Create adjacency matrix
N = len(gdf)
index_map = {code: i for i, code in enumerate(gdf["code"])}

A = np.zeros((N, N))

for _, row in neighbors.iterrows():
    i = index_map[row["code_left"]]
    j = index_map[row["code_right"]]
    A[i, j] = 1

print("Adjacency shape:", A.shape)
print("Average neighbors per node:", A.sum() / N)

Adjacency shape: (4719, 4719)
Average neighbors per node: 5.909302818393727


In [ ]:
# Add self loops
A_hat = A + np.eye(N)

# Degree matrix
D = np.diag(A_hat.sum(axis=1))

# Compute D^{-1/2}
D_inv_sqrt = np.linalg.inv(np.sqrt(D))

# Normalized adjacency
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt

print("Normalized adjacency ready.")

Normalized adjacency ready.


In [ ]:
# ==========================================
# SPATIO-TEMPORAL GCN (RESEARCH VERSION - FIXED)
# ==========================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# 1. Prepare Data
# -----------------------------

X_model = np.transpose(X, (0, 2, 1))  # (B, N, T)

train_size = 36

X_train = torch.tensor(X_model[:train_size], dtype=torch.float32)
Y_train = torch.tensor(Y[:train_size], dtype=torch.float32)

X_test = torch.tensor(X_model[train_size:], dtype=torch.float32)
Y_test = torch.tensor(Y[train_size:], dtype=torch.float32)

A_torch = torch.tensor(A_norm, dtype=torch.float32)

# -----------------------------
# 2. Baseline (Persistence)
# -----------------------------

Y_baseline = X_test[:, :, -1]
baseline_rmse = torch.sqrt(torch.mean((Y_baseline - Y_test)**2)).item()

# -----------------------------
# 3. Minimal Spatial Model
# -----------------------------

class SpatialGCN(nn.Module):
    def __init__(self, num_nodes):
        super(SpatialGCN, self).__init__()
        self.linear = nn.Linear(num_nodes, num_nodes)

    def forward(self, X, A):
        # X: (B, N, T)
        spatial_outputs = []

        for t in range(X.shape[2]):
            Xt = X[:, :, t]                 # (B, N)
            H_t = torch.matmul(Xt, A)       # (B, N) × (N, N) → (B, N)
            spatial_outputs.append(H_t)

        H = torch.stack(spatial_outputs, dim=2)  # (B, N, T)
        H = torch.mean(H, dim=2)                 # (B, N)

        out = self.linear(H)
        return out

model = SpatialGCN(num_nodes=X_train.shape[1])

optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# -----------------------------
# 4. Training
# -----------------------------

epochs = 200

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train, A_torch)
    loss = criterion(outputs, Y_train)

    loss.backward()
    optimizer.step()

# -----------------------------
# 5. Evaluation
# -----------------------------

model.eval()
with torch.no_grad():
    predictions = model(X_test, A_torch)
    model_rmse = torch.sqrt(torch.mean((predictions - Y_test)**2)).item()

improvement = (baseline_rmse - model_rmse) / baseline_rmse * 100

print("Baseline RMSE:", baseline_rmse)
print("GCN Model RMSE:", model_rmse)
print("Improvement over Baseline (%):", improvement)

Baseline RMSE: 1.2700642347335815
GCN Model RMSE: 1.4785242080688477
Improvement over Baseline (%): -16.413340966096435


In [ ]:
# ==========================================
# MULTI-DOMAIN SPATIO-TEMPORAL FRAMEWORK
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------
# Model Definition
# -----------------------------------------

class GCN_LSTM_Attention(nn.Module):
    def __init__(self, num_nodes, hidden_dim=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=num_nodes,
                            hidden_size=hidden_dim,
                            batch_first=True)
        self.attn = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, num_nodes)

    def forward(self, X, A):
        B, N, T = X.shape

        spatial_seq = []
        for t in range(T):
            Xt = X[:, :, t]
            Ht = torch.matmul(Xt, A)
            spatial_seq.append(Ht)

        spatial_seq = torch.stack(spatial_seq, dim=1)  # (B, T, N)
        lstm_out, _ = self.lstm(spatial_seq)

        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)

        return self.fc(context)

# -----------------------------------------
# Training + Prediction Function
# -----------------------------------------

def run_domain_experiment(data_matrix, A_norm, window=12, epochs=150):

    # data_matrix shape: (N, T)

    # Normalize per node
    mean = data_matrix.mean(axis=1, keepdims=True)
    std = data_matrix.std(axis=1, keepdims=True)
    std[std == 0] = 1
    data_norm = (data_matrix - mean) / std

    # Transpose to (T, N)
    data_ts = data_norm.T

    # Sliding windows
    X_list, Y_list = [], []
    for i in range(data_ts.shape[0] - window):
        X_list.append(data_ts[i:i+window])
        Y_list.append(data_ts[i+window])

    X = np.array(X_list)  # (B, window, N)
    Y = np.array(Y_list)  # (B, N)

    X = np.transpose(X, (0, 2, 1))  # (B, N, window)

    train_size = int(0.75 * len(X))

    X_train = torch.tensor(X[:train_size], dtype=torch.float32).to(device)
    Y_train = torch.tensor(Y[:train_size], dtype=torch.float32).to(device)
    X_test = torch.tensor(X[train_size:], dtype=torch.float32).to(device)
    Y_test = torch.tensor(Y[train_size:], dtype=torch.float32).to(device)

    A_torch = torch.tensor(A_norm, dtype=torch.float32).to(device)

    # Baseline
    Y_baseline = X_test[:, :, -1]
    baseline_rmse = torch.sqrt(torch.mean((Y_baseline - Y_test)**2)).item()

    # Model
    model = GCN_LSTM_Attention(num_nodes=X.shape[1]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.MSELoss()

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_train, A_torch)
        loss = criterion(preds, Y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        preds_test = model(X_test, A_torch)
        model_rmse = torch.sqrt(torch.mean((preds_test - Y_test)**2)).item()

    improvement = (baseline_rmse - model_rmse) / baseline_rmse * 100

    # -----------------------------------------
    # FUTURE FORECAST (Next timestep)
    # -----------------------------------------

    last_window = torch.tensor(
        X[-1:], dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        future_pred_norm = model(last_window, A_torch).cpu().numpy()[0]

    # Reverse normalization
    future_pred = future_pred_norm * std.flatten() + mean.flatten()

    return {
        "Baseline_RMSE": baseline_rmse,
        "Model_RMSE": model_rmse,
        "Improvement_%": improvement,
        "Future_Prediction": future_pred
    }

In [ ]:
crime_results = run_domain_experiment(crime_matrix.values, A_norm)
print("Crime Results:", crime_results)

Crime Results: {'Baseline_RMSE': 1.2807823419570923, 'Model_RMSE': 1.130478024482727, 'Improvement_%': 11.735352100864661, 'Future_Prediction': array([ 9.64421359, 18.7474176 ,  5.90800522, ...,  0.0690695 ,
       18.463379  ,  6.60109812])}


In [ ]:
# ==========================================
# BUILD CLEAN HOUSING MATRIX (LSOA × Time)
# ==========================================

import numpy as np
import pandas as pd

# 1️⃣ Keep only London LSOAs
london_codes = set(gdf["code"])

housing_london = housing[housing["LSOA code"].isin(london_codes)].copy()

print("London housing rows:", housing_london.shape[0])

# 2️⃣ Select only recent period (2010 onward to match crime window)
cols = housing_london.columns.tolist()

recent_cols = [c for c in cols if "2010" in c or
                                   "2011" in c or
                                   "2012" in c or
                                   "2013" in c or
                                   "2014" in c or
                                   "2015" in c or
                                   "2016" in c or
                                   "2017" in c or
                                   "2018" in c or
                                   "2019" in c or
                                   "2020" in c or
                                   "2021" in c or
                                   "2022" in c or
                                   "2023" in c]

housing_recent = housing_london[["LSOA code"] + recent_cols].copy()

# 3️⃣ Clean numeric values
for col in recent_cols:
    housing_recent[col] = (
        housing_recent[col]
        .replace(":", np.nan)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

# 4️⃣ Reindex to match spatial order
housing_recent = housing_recent.set_index("LSOA code")
housing_recent = housing_recent.reindex(gdf["code"])

# 5️⃣ Handle missing values (forward fill, then backfill)
housing_recent = housing_recent.fillna(method="ffill", axis=1)
housing_recent = housing_recent.fillna(method="bfill", axis=1)
housing_recent = housing_recent.fillna(0)

# 6️⃣ Final matrix
housing_matrix = housing_recent.values  # (N × T)

print("Housing matrix shape:", housing_matrix.shape)

London housing rows: 4719
Housing matrix shape: (4719, 53)


/tmp/ipykernel_5836/28011162.py:49: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  housing_recent = housing_recent.fillna(method="ffill", axis=1)
/tmp/ipykernel_5836/28011162.py:50: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  housing_recent = housing_recent.fillna(method="bfill", axis=1)


In [ ]:
housing_results = run_domain_experiment(housing_matrix, A_norm, window=8)
print("Housing Results:", housing_results)

Housing Results: {'Baseline_RMSE': 0.516895592212677, 'Model_RMSE': 0.8584328293800354, 'Improvement_%': -66.07470489452977, 'Future_Prediction': array([360342.19704578, 375412.3330524 , 363573.46609921, ...,
       487092.50860473, 370290.71606459, 484793.69293661])}


In [ ]:
# ==========================================
# AGGREGATE DAILY → MONTHLY
# ==========================================

transport = crime_hist.copy()

transport["TravelDate"] = pd.to_datetime(transport["TravelDate"])
transport["YearMonth"] = transport["TravelDate"].dt.to_period("M")

transport["TotalTap"] = transport["EntryTapCount"] + transport["ExitTapCount"]

monthly_station = (
    transport
    .groupby(["Station", "YearMonth"])["TotalTap"]
    .sum()
    .reset_index()
)

print(monthly_station.head())

          Station YearMonth  TotalTap
0  Abbey Road DLR   2019-01     45557
1  Abbey Road DLR   2019-02     45566
2  Abbey Road DLR   2019-03     51207
3  Abbey Road DLR   2019-04     47386
4  Abbey Road DLR   2019-05     49751


In [ ]:
# ==========================================
# FINAL TRANSPORT PIPELINE (DRIVE VERSION)
# ==========================================

import pandas as pd
import numpy as np
import geopandas as gpd

# -----------------------------
# 1️⃣ Load Station Coordinates
# -----------------------------

station_df = pd.read_csv("/content/drive/MyDrive/nndl/station_coords.csv")

print("Station file columns:", station_df.columns)

# Auto-detect columns
station_col = [c for c in station_df.columns if "station" in c.lower()][0]
lat_col = [c for c in station_df.columns if "lat" in c.lower()][0]
lon_col = [c for c in station_df.columns if "lon" in c.lower()][0]

print("Using columns:", station_col, lat_col, lon_col)

# -----------------------------
# 2️⃣ Convert to GeoDataFrame
# -----------------------------

stations_geo = gpd.GeoDataFrame(
    station_df,
    geometry=gpd.points_from_xy(
        station_df[lon_col],
        station_df[lat_col]
    ),
    crs="EPSG:4326"
)

gdf_lsoa = gdf.to_crs("EPSG:4326")

# -----------------------------
# 3️⃣ Spatial Join
# -----------------------------

stations_lsoa = gpd.sjoin(
    stations_geo,
    gdf_lsoa[["code", "geometry"]],
    predicate="within",
    how="left"
)

print("Stations mapped to LSOA:", stations_lsoa["code"].notna().sum())

# -----------------------------
# 4️⃣ Load Transport Data
# -----------------------------

transport = pd.read_csv("/content/drive/MyDrive/nndl/StationFootfall_Merged_2019-2023.csv")

transport["TravelDate"] = pd.to_datetime(transport["TravelDate"])
transport["YearMonth"] = transport["TravelDate"].dt.to_period("M")

transport["TotalTap"] = (
    transport["EntryTapCount"] +
    transport["ExitTapCount"]
)

# -----------------------------
# 5️⃣ Monthly Aggregation
# -----------------------------

monthly_station = (
    transport
    .groupby(["Station", "YearMonth"])["TotalTap"]
    .sum()
    .reset_index()
)

# -----------------------------
# 6️⃣ Merge Station → LSOA
# -----------------------------

monthly_station = monthly_station.merge(
    stations_lsoa[[station_col, "code"]],
    left_on="Station",
    right_on=station_col,
    how="left"
)

monthly_station = monthly_station.dropna(subset=["code"])

# -----------------------------
# 7️⃣ Aggregate to LSOA
# -----------------------------

monthly_lsoa = (
    monthly_station
    .groupby(["code", "YearMonth"])["TotalTap"]
    .sum()
    .reset_index()
)

# -----------------------------
# 8️⃣ Pivot to Matrix
# -----------------------------

transport_matrix_df = monthly_lsoa.pivot(
    index="code",
    columns="YearMonth",
    values="TotalTap"
).fillna(0)

transport_matrix_df = transport_matrix_df.reindex(gdf_lsoa["code"]).fillna(0)

transport_matrix = transport_matrix_df.values

print("Transport matrix shape:", transport_matrix.shape)

Station file columns: Index(['Station', 'latitude', 'longitude', 'address'], dtype='object')
Using columns: Station latitude longitude
Stations mapped to LSOA: 398
Transport matrix shape: (4719, 60)


In [ ]:
transport_results = run_domain_experiment(
    transport_matrix,
    A_norm,
    window=12,
    epochs=150
)

print("Transport Results:", transport_results)

Transport Results: {'Baseline_RMSE': 0.10774853080511093, 'Model_RMSE': 0.23735374212265015, 'Improvement_%': -120.28489887436268, 'Future_Prediction': array([ 0.22878487,  0.01083139,  0.02659818, ..., -0.01684172,
       -0.07103683,  0.00987401])}


In [ ]:
# ==========================================
# ATTACH CRIME PREDICTION TO MAP
# ==========================================

# Get future crime prediction (next timestep)
crime_future = crime_results["Future_Prediction"]

# Add to GeoDataFrame
gdf_demo = gdf.copy()
gdf_demo["Predicted_Crime"] = crime_future

print(gdf_demo[["code", "Predicted_Crime"]].head())

        code  Predicted_Crime
0  E01000037         9.644214
1  E01033729        18.747418
2  E01000038         5.908005
3  E01033730        -0.059933
4  E01000039         6.951102


In [ ]:
# ==========================================
# FINAL FEATURE-ENHANCED MULTI-DOMAIN TRAINING
# ==========================================

import numpy as np
import pandas as pd
import torch
import random

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

# -----------------------------
# Ensure Population exists
# -----------------------------
if "Population" not in gdf_demo.columns:
    if "Half densi" in gdf_demo.columns and "Area" in gdf_demo.columns:
        gdf_demo["Population"] = (
            2 * gdf_demo["Half densi"].fillna(0) *
            (gdf_demo["Area"].fillna(0) / 1_000_000)
        ).round().astype(int)
    else:
        gdf_demo["Population"] = 0

# -----------------------------
# Ensure Num_Stations exists
# -----------------------------
if "Num_Stations" not in gdf_demo.columns:
    station_counts = (
        stations_lsoa.groupby("code")
        .size()
        .reset_index(name="Num_Stations")
    )

    gdf_demo = gdf_demo.merge(
        station_counts,
        on="code",
        how="left"
    )

    gdf_demo["Num_Stations"] = gdf_demo["Num_Stations"].fillna(0).astype(int)

# -----------------------------
# Convert matrices to numpy
# -----------------------------
crime_np = crime_matrix.values if isinstance(crime_matrix, pd.DataFrame) else crime_matrix
housing_np = housing_matrix.values if isinstance(housing_matrix, pd.DataFrame) else housing_matrix
transport_np = transport_matrix.values if isinstance(transport_matrix, pd.DataFrame) else transport_matrix

# -----------------------------
# External Features
# -----------------------------
external_features = np.column_stack([
    gdf_demo["Population"].values,
    gdf_demo["Num_Stations"].values,
    crime_np[:, -1]     # latest crime value
])

# Normalize
ext_mean = external_features.mean(axis=0)
ext_std = external_features.std(axis=0)
external_features = (external_features - ext_mean) / (ext_std + 1e-6)

# -----------------------------
# Housing Trend Removal
# -----------------------------
housing_diff = np.diff(housing_np, axis=1)

# -----------------------------
# Transport Log Transform
# -----------------------------
transport_log = np.log1p(transport_np)

# -----------------------------
# Seasonality Feature
# -----------------------------
months = np.arange(transport_log.shape[1]) % 12
season_feature = np.sin(2 * np.pi * months / 12)

season_feature = np.tile(season_feature, (transport_log.shape[0], 1))

transport_enhanced = transport_log + 0.1 * season_feature

# -----------------------------
# Evaluation Function
# -----------------------------
runs = 3
results_summary = {}

def evaluate_domain(data_matrix, window, epochs):

    rmse_list = []
    baseline_list = []

    for r in range(runs):

        set_seed(42 + r)

        results = run_domain_experiment(
            np.array(data_matrix),
            A_norm,
            window=window,
            epochs=80
        )

        rmse_list.append(results["Model_RMSE"])
        baseline_list.append(results["Baseline_RMSE"])

    base = np.mean(baseline_list)
    model = np.mean(rmse_list)

    return {
        "Baseline_RMSE": base,
        "Model_RMSE": model,
        "Improvement_%": ((base - model) / base) * 100
    }

# -----------------------------
# Crime
# -----------------------------
results_summary["Crime"] = evaluate_domain(
    crime_np,
    window=12,
    epochs=80
)

# -----------------------------
# Housing
# -----------------------------
results_summary["Housing"] = evaluate_domain(
    housing_diff,
    window=6,
    epochs=80
)

# -----------------------------
# Transport
# -----------------------------
results_summary["Transport"] = evaluate_domain(
    transport_enhanced,
    window=8,
    epochs=80
)

# -----------------------------
# Display Results
# -----------------------------
results_df = pd.DataFrame(results_summary).T.round(4)

print("\n===== FEATURE-ENHANCED MULTI-DOMAIN RESULTS =====\n")
print(results_df)


===== FEATURE-ENHANCED MULTI-DOMAIN RESULTS =====

           Baseline_RMSE  Model_RMSE  Improvement_%
Crime             1.2808      1.0915        14.7768
Housing           1.7512      1.3355        23.7354
Transport         0.5046      0.3863        23.4339


In [ ]:
import geopandas as gpd

gdf_full = gpd.read_file(
    "/content/drive/MyDrive/nndl/LSOA_2011_EW_BFC_V3.shp"
)

print("Full shapefile shape:", gdf_full.shape)
print(gdf_full.columns)

Full shapefile shape: (34753, 9)
Index(['LSOA11CD', 'LSOA11NM', 'BNG_E', 'BNG_N', 'LONG_', 'LAT', 'Shape_Leng',
       'GlobalID', 'geometry'],
      dtype='object')


In [ ]:
london_codes = set(gdf["code"])

gdf_london_full = gdf_full[
    gdf_full["LSOA11CD"].isin(london_codes)
].copy()

print("London LSOAs in full shapefile:", gdf_london_full.shape)

London LSOAs in full shapefile: (4719, 9)


In [ ]:
gdf_london_full = gdf_london_full.merge(
    gdf[["code"]],
    left_on="LSOA11CD",
    right_on="code",
    how="left"
)

gdf_london_full["Predicted_Crime"] = crime_results["Future_Prediction"]

In [ ]:
print(gdf_london_full.columns)

Index(['LSOA11CD', 'LSOA11NM', 'BNG_E', 'BNG_N', 'LONG_', 'LAT', 'Shape_Leng',
       'GlobalID', 'geometry', 'code', 'Predicted_Crime'],
      dtype='object')


In [ ]:
# ==========================================
# COUNT NUMBER OF STATIONS PER LSOA
# ==========================================

station_counts = (
    stations_lsoa.groupby("code")
    .size()
    .reset_index(name="Num_Stations")
)

gdf_demo = gdf.copy()

gdf_demo = gdf_demo.merge(
    station_counts,
    on="code",
    how="left"
)

gdf_demo["Num_Stations"] = gdf_demo["Num_Stations"].fillna(0)

print(gdf_demo[["code", "Num_Stations"]].head())

        code  Num_Stations
0  E01000037           0.0
1  E01033729           0.0
2  E01000038           0.0
3  E01033730           0.0
4  E01000039           0.0


In [ ]:
# Latest housing price
latest_housing = housing_matrix[:, -1]

gdf_demo["Latest_Housing"] = latest_housing

In [ ]:
gdf_demo["Predicted_Crime"] = crime_results["Future_Prediction"]

In [ ]:
gdf_demo["Population"] = (
    2 * gdf_demo["Half densi"] *
    (gdf_demo["Area"] / 1_000_000)
)

gdf_demo["Population"] = gdf_demo["Population"].round().astype(int)

In [ ]:
print(gdf_demo["Population"].describe())

count    4719.000000
mean     1795.777495
std       392.241031
min       817.000000
25%      1572.000000
50%      1722.000000
75%      1924.500000
max      8744.000000
Name: Population, dtype: float64


In [ ]:
gdf_demo["Latest_Housing"] = gdf_demo["Latest_Housing"].astype(int)
gdf_demo["Population"] = gdf_demo["Population"].astype(int)

In [ ]:
# NOTE: This cell contains DUPLICATE logic for calculating 'Population'.
# This feature was already calculated and assigned in cell cCX4hOIuAcDu.
if "Half densi" in gdf_demo.columns and "Area" in gdf_demo.columns:
    gdf_demo["Population"] = (
        2 * gdf_demo["Half densi"].fillna(0) *
        (gdf_demo["Area"].fillna(0) / 1_000_000)
    ).round().astype(int)
else:
    gdf_demo["Population"] = 0

In [ ]:
# ==========================================
# INTERACTIVE LONDON LSOA MAP (FIXED)
# ==========================================

import folium
from branca.colormap import linear
import numpy as np
import json # Keep json import for general safety, even if not directly used in GeoJson(gdf_demo)

# -----------------------------
# 1. PREPARE DATA
# -----------------------------
gdf_demo = gdf.copy()

# Ensure 'code' column is present for merging/indexing
if "code" not in gdf_demo.columns:
    gdf_demo["code"] = gdf_demo.index

# Attach predictions from the stable model
gdf_demo["Predicted_Crime"] = crime_pred
gdf_demo["Predicted_Housing_Change"] = housing_pred # Renaming for clarity as housing_pred is a diff
gdf_demo["Predicted_Transport"] = transport_pred

# Attach latest observed housing value (matches user's tooltip request for "Latest Housing Price")
# 'latest_housing' variable is available from previous cells (uPYyrrK8SRYX)
gdf_demo["Latest_Housing"] = latest_housing

# Calculate Population (using robust logic from previous cells)
if "Half densi" in gdf_demo.columns and "Area" in gdf_demo.columns:
    gdf_demo["Population"] = (
        2 * gdf_demo["Half densi"].fillna(0) *
        (gdf_demo["Area"].fillna(0) / 1_000_000)
    ).round().astype(int)
else:
    gdf_demo["Population"] = 0

# Calculate Num_Stations (using robust logic from previous cells)
# Ensure station_counts is available from previous execution if not already merged
if 'stations_lsoa' in locals() and 'Num_Stations' not in gdf_demo.columns:
    station_counts = (
        stations_lsoa.groupby("code")
        .size()
        .reset_index(name="Num_Stations")
    )
    gdf_demo = gdf_demo.merge(
        station_counts,
        on="code",
        how="left"
    )
    gdf_demo["Num_Stations"] = gdf_demo["Num_Stations"].fillna(0).astype(int)
elif "Num_Stations" not in gdf_demo.columns:
    gdf_demo["Num_Stations"] = 0 # Fallback if stations_lsoa is not found


# -----------------------------
# Clean numeric formatting for display
# -----------------------------
gdf_demo["Predicted_Crime"] = gdf_demo["Predicted_Crime"].round(2)
gdf_demo["Latest_Housing"] = gdf_demo["Latest_Housing"].round().astype(int)
gdf_demo["Population"] = gdf_demo["Population"].round().astype(int)
gdf_demo["Num_Stations"] = gdf_demo["Num_Stations"].astype(int)


# -----------------------------
# Create Map
# -----------------------------
m = folium.Map(location=[51.5074, -0.1278], zoom_start=10)

colormap = linear.YlOrRd_09.scale(
    gdf_demo["Predicted_Crime"].min(),
    gdf_demo["Predicted_Crime"].max()
)

colormap.caption = "Predicted Crime (Next Month)"
colormap.add_to(m)

# -----------------------------
# Add LSOA polygons
# -----------------------------
folium.GeoJson(
    gdf_demo, # Use GeoDataFrame directly as per user's working code
    style_function=lambda feature: {
        "fillColor": colormap(feature["properties"]["Predicted_Crime"]),
        "color": "black",
        "weight": 0.2,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "code",
            "Predicted_Crime",
            "Population",
            "Latest_Housing", # Use Latest_Housing for the actual price
            "Num_Stations"
        ],
        aliases=[
            "LSOA Code:",
            "Predicted Crime:",
            "Population:",
            "Latest Housing Price (£):",
            "Number of Stations:"
        ],
        localize=True
    )
).add_to(m)

# -----------------------------
# Show map
# -----------------------------
m


In [ ]:
print(gdf_demo["Latest_Housing"].describe())

count    4.719000e+03
mean     6.137527e+05
std      3.911414e+05
min      0.000000e+00
25%      4.175000e+05
50%      5.200000e+05
75%      6.750000e+05
max      6.600000e+06
Name: Latest_Housing, dtype: float64


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()

# -------------------------
# SEQUENCE CREATION
# -------------------------
def create_sequences(data, window):
    X, Y = [], []
    for i in range(data.shape[1] - window):
        X.append(data[:, i:i+window])
        Y.append(data[:, i+window])
    return np.array(X), np.array(Y)

# -------------------------
# MODEL
# -------------------------
class Model(nn.Module):
    def __init__(self, ext_dim, domain):
        super().__init__()
        self.domain = domain

        self.temp_embed = nn.Linear(1, 64)
        self.ext_embed = nn.Linear(ext_dim, 64)

        # GCN only for non-crime
        if domain != "Crime":
            self.gcn1 = nn.Linear(64, 64)
            self.gcn2 = nn.Linear(64, 64)

        self.gate = nn.Linear(128, 64)

        # Domain-specific LSTM
        if domain == "Transport":
            hidden = 64
        else:
            hidden = 32

        self.lstm = nn.LSTM(64, hidden, batch_first=True)
        self.attn = nn.Linear(hidden, 1)

        self.fc = nn.Sequential(
            nn.Linear(hidden, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        self.dropout = nn.Dropout(0.3)

    def forward(self, X, X_ext, A):
        B, N, T = X.shape

        X = X.unsqueeze(-1)
        X_temp = self.temp_embed(X)

        X_ext = self.ext_embed(X_ext)
        X_ext = X_ext.unsqueeze(0).unsqueeze(2).repeat(B, 1, T, 1)

        # Fusion
        fusion = torch.cat([X_temp, X_ext], dim=-1)
        gate = torch.sigmoid(self.gate(fusion))
        X = gate * X_temp + (1 - gate) * X_ext

        # -------- DOMAIN-SPECIFIC SPATIAL --------
        if self.domain != "Crime":
            X = X.permute(0,2,1,3).reshape(B*T, N, 64)
            X = torch.relu(self.gcn1(A @ X))
            X = torch.relu(self.gcn2(A @ X))
            X = X.reshape(B, T, N, 64)
        else:
            X = X.reshape(B, T, N, 64)  # skip GCN

        # LSTM
        X = X.permute(0,2,1,3).reshape(B*N, T, 64)
        out, _ = self.lstm(X)

        out = self.dropout(out)

        # Attention
        attn = torch.softmax(self.attn(out), dim=1)
        context = torch.sum(attn * out, dim=1)

        return self.fc(context).reshape(B, N)

# -------------------------
# TRAIN FUNCTION
# -------------------------
def train_model(data, A, ext_features, domain, window, epochs=20):

    # -------- DOMAIN NORMALIZATION --------
    mean = data.mean(axis=1, keepdims=True)
    std = data.std(axis=1, keepdims=True)
    data = (data - mean) / (std + 1e-6)

    X, Y = create_sequences(data, window)

    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)

    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    Y_train, Y_test = Y[:split], Y[split:]

    model = Model(ext_features.shape[1], domain).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
    loss_fn = nn.MSELoss()

    ext_features = torch.tensor(ext_features, dtype=torch.float32).to(device)
    A = torch.tensor(A, dtype=torch.float32).to(device)

    batch_nodes = 256

    print("🚀 Training...")

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for i in range(0, data.shape[0], batch_nodes):
            idx = slice(i, i+batch_nodes)

            Xb = X_train[:, idx, :].to(device)
            Yb = Y_train[:, idx].to(device)
            Extb = ext_features[idx]
            Ab = A[idx][:, idx]

            pred = model(Xb, Extb, Ab)
            loss = loss_fn(pred, Yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch} | Loss: {total_loss:.4f}")

    # -------- EVALUATION --------
    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, data.shape[0], batch_nodes):
            idx = slice(i, i+batch_nodes)

            Xb = X_test[:, idx, :].to(device)
            Extb = ext_features[idx]
            Ab = A[idx][:, idx]

            preds.append(model(Xb, Extb, Ab))

    pred = torch.cat(preds, dim=1)

    # Fair baseline
    baseline = X_test[:, :, -3:].mean(dim=2).to(device)

    Y_test = Y_test.to(device)

    rmse_model = torch.sqrt(torch.mean((Y_test - pred) ** 2)).item()
    rmse_base = torch.sqrt(torch.mean((Y_test - baseline) ** 2)).item()

    return rmse_model, rmse_base

# -------------------------
# RUN DOMAINS
# -------------------------
results = {}

domains = {
    "Crime": (crime_np, 8),        # shorter window
    "Housing": (housing_diff, 6),
    "Transport": (transport_enhanced, 8)
}

for name, (data, window) in domains.items():
    print(f"\n🔥 Training {name}...\n")

    rmse_model, rmse_base = train_model(
        data,
        A_norm,
        external_features,
        domain=name,
        window=window
    )

    results[name] = {
        "Baseline_RMSE": rmse_base,
        "Model_RMSE": rmse_model,
        "Improvement_%": ((rmse_base - rmse_model) / rmse_base) * 100
    }

# -------------------------
# FINAL RESULTS
# -------------------------
df = pd.DataFrame(results).T.round(4)

print("\n===== FINAL RESULTS (STABLE) =====\n")
print(df)


🔥 Training Crime...

🚀 Training...
Epoch 0 | Loss: 18.3682
Epoch 1 | Loss: 18.2525
Epoch 2 | Loss: 18.2422
Epoch 3 | Loss: 18.2381
Epoch 4 | Loss: 18.2383
Epoch 5 | Loss: 18.2387
Epoch 6 | Loss: 18.2386
Epoch 7 | Loss: 18.2384
Epoch 8 | Loss: 18.2368
Epoch 9 | Loss: 18.2369
Epoch 10 | Loss: 18.2371
Epoch 11 | Loss: 18.2370
Epoch 12 | Loss: 18.2375
Epoch 13 | Loss: 18.2349
Epoch 14 | Loss: 18.2354
Epoch 15 | Loss: 18.2351
Epoch 16 | Loss: 18.2344
Epoch 17 | Loss: 18.2341
Epoch 18 | Loss: 18.2346
Epoch 19 | Loss: 18.2336

🔥 Training Housing...

🚀 Training...
Epoch 0 | Loss: 18.4891
Epoch 1 | Loss: 18.2738
Epoch 2 | Loss: 18.2422
Epoch 3 | Loss: 18.1469
Epoch 4 | Loss: 18.1052
Epoch 5 | Loss: 18.0784
Epoch 6 | Loss: 18.0528
Epoch 7 | Loss: 18.0407
Epoch 8 | Loss: 18.0337
Epoch 9 | Loss: 18.0153
Epoch 10 | Loss: 18.0128
Epoch 11 | Loss: 18.0061
Epoch 12 | Loss: 17.9876
Epoch 13 | Loss: 17.9692
Epoch 14 | Loss: 17.9532
Epoch 15 | Loss: 17.9403
Epoch 16 | Loss: 17.9172
Epoch 17 | Loss: 17.8

In [ ]:
print("===== Model Performance Summary ====")
for domain, metrics in results.items():
    print(f"\nDomain: {domain}")
    print(f"  Baseline RMSE: {metrics['Baseline_RMSE']:.4f}")
    print(f"  Model RMSE:    {metrics['Model_RMSE']:.4f}")
    print(f"  Improvement %: {metrics['Improvement_%']:.2f}%")

===== Model Performance Summary ====

Domain: Crime
  Baseline RMSE: 1.0664
  Model RMSE:    0.9661
  Improvement %: 9.40%

Domain: Housing
  Baseline RMSE: 1.4263
  Model RMSE:    1.2102
  Improvement %: 15.15%

Domain: Transport
  Baseline RMSE: 0.9101
  Model RMSE:    0.3059
  Improvement %: 66.39%


In [ ]:
# NOTE: This cell is a DUPLICATE of cell 8d2bc7de.
# It prints the exact same Model Performance Summary.
print("===== Model Performance Summary (Duplicate of cell 8d2bc7de) ====")
for domain, metrics in results.items():
    print(f"\nDomain: {domain}")
    print(f"  Baseline RMSE: {metrics['Baseline_RMSE']:.4f}")
    print(f"  Model RMSE:    {metrics['Model_RMSE']:.4f}")
    print(f"  Improvement %: {metrics['Improvement_%']:.2f}%")

===== Model Performance Summary ====

Domain: Crime
  Baseline RMSE: 1.0664
  Model RMSE:    0.9661
  Improvement %: 9.40%

Domain: Housing
  Baseline RMSE: 1.4263
  Model RMSE:    1.2102
  Improvement %: 15.15%

Domain: Transport
  Baseline RMSE: 0.9101
  Model RMSE:    0.3059
  Improvement %: 66.39%


In [ ]:
# =========================================
# FINAL STABLE MODEL + SAVE/LOAD (ONE CELL)
# =========================================
# NOTE: This cell encapsulates the complete multi-domain GCN-LSTM architecture,
# training, and denormalized forecasting pipeline. It operates independently
# by loading precomputed arrays if they exist, avoiding redundant GPU/CPU retraining.

import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import os

# -------------------------
# DRIVE PATH
# -------------------------
BASE_PATH = "/content/drive/MyDrive/nndl/"

# -------------------------
# DEVICE (CPU SAFE)
# -------------------------
device = torch.device("cpu")
torch.set_num_threads(4)

# -------------------------
# CHECK IF PREDICTIONS EXIST
# -------------------------
if os.path.exists(BASE_PATH + "crime_pred.npy"):

    print("✅ Loading saved predictions...")

    crime_pred = np.load(BASE_PATH + "crime_pred.npy")
    housing_pred = np.load(BASE_PATH + "housing_pred.npy")
    transport_pred = np.load(BASE_PATH + "transport_pred.npy")

else:

    print("🚀 No saved data found. Training model...")

    # -------------------------
    # SEQUENCE CREATION
    # -------------------------
    def create_sequences(data, window):
        # data shape: (N, T)
        # Transpose to (T, N) for slicing
        data_transposed = data.T # (T, N)
        X, Y = [], []
        for i in range(data_transposed.shape[0] - window):
            X.append(data_transposed[i:i+window].T) # (N, window)
            Y.append(data_transposed[i+window])     # (N,)
        return np.array(X), np.array(Y) # X: (num_sequences, N, window), Y: (num_sequences, N)

    # -------------------------
    # MODEL DEFINITION
    # -------------------------
    class Model(nn.Module):
        def __init__(self, ext_dim, domain):
            super().__init__()
            self.domain = domain

            self.temp_embed = nn.Linear(1, 64)
            self.ext_embed = nn.Linear(ext_dim, 64)

            if domain != "Crime":
                self.gcn1 = nn.Linear(64, 64)
                self.gcn2 = nn.Linear(64, 64)

            self.gate = nn.Linear(128, 64)

            hidden = 64 if domain == "Transport" else 32

            self.lstm = nn.LSTM(64, hidden, batch_first=True)
            self.attn = nn.Linear(hidden, 1)

            self.fc = nn.Sequential(
                nn.Linear(hidden, 16),
                nn.ReLU(),
                nn.Linear(16, 1)
            )

            self.dropout = nn.Dropout(0.3)

        def forward(self, X, X_ext, A):
            B, N, T = X.shape

            X = X.unsqueeze(-1) # (B, N, T, 1)
            X_temp = self.temp_embed(X)

            X_ext = self.ext_embed(X_ext)
            X_ext = X_ext.unsqueeze(0).unsqueeze(2).repeat(B, 1, T, 1)

            fusion = torch.cat([X_temp, X_ext], dim=-1)
            gate = torch.sigmoid(self.gate(fusion))
            X = gate * X_temp + (1 - gate) * X_ext # (B, N, T, 64)

            # GCN (Applied conditionally based on Domain)
            if self.domain != "Crime":
                X_gcn_input = X.permute(0, 2, 1, 3).reshape(B * T, N, 64)
                X_gcn_output = torch.relu(self.gcn1(torch.matmul(A, X_gcn_input)))
                X_gcn_output = torch.relu(self.gcn2(torch.matmul(A, X_gcn_output)))
                X = X_gcn_output.reshape(B, T, N, 64)
            else:
                X = X.permute(0, 2, 1, 3) # (B, T, N, 64)

            # LSTM (chunked processing for memory efficiency)
            outputs = []
            chunk = 128

            for i in range(0, N, chunk):
                X_chunk = X[:, :, i:i+chunk, :]
                Bc, T_seq, Nc, F = X_chunk.shape
                X_chunk = X_chunk.reshape(Bc * Nc, T_seq, F)

                out_chunk, _ = self.lstm(X_chunk)
                outputs.append(out_chunk)

            out = torch.cat(outputs, dim=0) # (B*N, T_seq, hidden_dim)
            out = self.dropout(out)

            attn = torch.softmax(self.attn(out), dim=1)
            context = torch.sum(attn * out, dim=1) # (B*N, hidden_dim)

            return self.fc(context).reshape(B, N) # (B, N)

    # -------------------------
    # TRAIN FUNCTION
    # -------------------------
    def train_model(data, A, ext_features, domain, window):
        original_mean = data.mean(axis=1, keepdims=True)
        original_std = data.std(axis=1, keepdims=True)
        original_std[original_std == 0] = 1 # Avoid division by zero

        mean = data.mean(axis=1, keepdims=True)
        std = data.std(axis=1, keepdims=True)
        data_norm = (data - mean) / (std + 1e-6)

        X_sequences, Y_sequences = create_sequences(data_norm, window)

        X = torch.tensor(X_sequences, dtype=torch.float32)
        Y = torch.tensor(Y_sequences, dtype=torch.float32)

        split = int(0.8 * len(X))
        X_train, X_test = X[:split], X[split:]
        Y_train, Y_test = Y[:split], Y[split:]

        model = Model(ext_features.shape[1], domain).to(device)

        lr = 0.003 if domain == "Crime" else 0.001
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        loss_fn = nn.HuberLoss()

        X_ext_torch = torch.tensor(ext_features, dtype=torch.float32).to(device)
        A_torch = torch.tensor(A, dtype=torch.float32).to(device)

        batch_size = 4
        epochs = 30 if domain == "Crime" else 15

        print(f"🚀 Training {domain} Model...")

        for epoch in range(epochs):
            model.train()
            total_loss = 0

            for i in range(0, X_train.shape[0], batch_size):
                Xb = X_train[i:i+batch_size].to(device)
                Yb = Y_train[i:i+batch_size].to(device)

                pred = model(Xb, X_ext_torch, A_torch)
                loss = loss_fn(pred, Yb)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            if epoch % 5 == 0:
                print(f"Epoch {epoch} | Loss: {total_loss:.4f}")

        # -------- FUTURE FORECAST --------
        model.eval()
        with torch.no_grad():
            last_window_data_norm = data_norm[:, -window:]
            last_window_input = torch.tensor(
                last_window_data_norm, dtype=torch.float32
            ).unsqueeze(0).to(device)

            future_pred_norm = model(last_window_input, X_ext_torch, A_torch).cpu().numpy()[0]

        future_pred = future_pred_norm * original_std.flatten() + original_mean.flatten()
        return future_pred

    # -------------------------
    # TRAIN ALL DOMAINS
    # -------------------------
    crime_pred = train_model(crime_np, A_norm, external_features, "Crime", 8)
    housing_pred = train_model(housing_diff, A_norm, external_features, "Housing", 6)
    transport_pred = train_model(transport_enhanced, A_norm, external_features, "Transport", 8)

    # -------------------------
    # SAVE RESULTS
    # -------------------------
    np.save(BASE_PATH + "crime_pred.npy", crime_pred)
    np.save(BASE_PATH + "housing_pred.npy", housing_pred)
    np.save(BASE_PATH + "transport_pred.npy", transport_pred)

    print("✅ Predictions saved to Drive")

# -------------------------
# FINAL VERIFICATION
# -------------------------
print("\n===== READY FOR MAP =====")
print("Crime Shape:", crime_pred.shape)
print("Housing Shape:", housing_pred.shape)
print("Transport Shape:", transport_pred.shape)

🚀 No saved data found. Training model...
🚀 Training...
Epoch 0 | Loss: 4.4664
Epoch 5 | Loss: 4.4206
Epoch 10 | Loss: 4.4201
Epoch 15 | Loss: 4.4200
Epoch 20 | Loss: 4.4200
Epoch 25 | Loss: 4.4199
🚀 Training...
Epoch 0 | Loss: 3.3138
Epoch 5 | Loss: 3.1504
Epoch 10 | Loss: 3.1493
🚀 Training...
Epoch 0 | Loss: 5.0066
Epoch 5 | Loss: 4.6342
Epoch 10 | Loss: 4.6094
✅ Predictions saved to Drive

===== READY FOR MAP =====
Crime: (4719,)
Housing: (4719,)
Transport: (4719,)


In [ ]:
# ==========================================
# INTERACTIVE LONDON LSOA MAP (FIXED & COMMENTED)
# ==========================================
# NOTE: This cell acts as the final mapping interface. It utilizes variables
# generated from the stable prediction pipeline (Sos6V4pO15In) and previous data layers.

import folium
from branca.colormap import linear
import numpy as np
import json

# -----------------------------
# 1. PREPARE DATA
# -----------------------------
gdf_demo = gdf.copy()

# Ensure 'code' column is present for merging/indexing
if "code" not in gdf_demo.columns:
    gdf_demo["code"] = gdf_demo.index

# Attach predictions from the final stable model
gdf_demo["Predicted_Crime"] = crime_pred
gdf_demo["Predicted_Housing_Change"] = housing_pred
gdf_demo["Predicted_Transport"] = transport_pred

# Attach latest observed housing value (computed in cell uPYyrrK8SRYX)
gdf_demo["Latest_Housing"] = latest_housing

# Calculate Population
if "Half densi" in gdf_demo.columns and "Area" in gdf_demo.columns:
    gdf_demo["Population"] = (
        2 * gdf_demo["Half densi"].fillna(0) *
        (gdf_demo["Area"].fillna(0) / 1_000_000)
    ).round().astype(int)
else:
    gdf_demo["Population"] = 0

# Calculate Station Counts from spatial join information
if 'stations_lsoa' in locals() and 'Num_Stations' not in gdf_demo.columns:
    station_counts = (
        stations_lsoa.groupby("code")
        .size()
        .reset_index(name="Num_Stations")
    )
    gdf_demo = gdf_demo.merge(
        station_counts,
        on="code",
        how="left"
    )
    gdf_demo["Num_Stations"] = gdf_demo["Num_Stations"].fillna(0).astype(int)
elif "Num_Stations" not in gdf_demo.columns:
    gdf_demo["Num_Stations"] = 0

# -----------------------------
# Clean numeric formatting for display
# -----------------------------
gdf_demo["Predicted_Crime"] = gdf_demo["Predicted_Crime"].round(2)
gdf_demo["Latest_Housing"] = gdf_demo["Latest_Housing"].round().astype(int)
gdf_demo["Population"] = gdf_demo["Population"].round().astype(int)
gdf_demo["Num_Stations"] = gdf_demo["Num_Stations"].astype(int)

# -----------------------------
# Create Map
# -----------------------------
m = folium.Map(location=[51.5074, -0.1278], zoom_start=10)

colormap = linear.YlOrRd_09.scale(
    gdf_demo["Predicted_Crime"].min(),
    gdf_demo["Predicted_Crime"].max()
)

colormap.caption = "Predicted Crime (Next Month)"
colormap.add_to(m)

# -----------------------------
# Add LSOA polygons
# -----------------------------
folium.GeoJson(
    gdf_demo,
    style_function=lambda feature: {
        "fillColor": colormap(feature["properties"]["Predicted_Crime"]),
        "color": "black",
        "weight": 0.2,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "code",
            "Predicted_Crime",
            "Population",
            "Latest_Housing"
        ],
        aliases=[
            "LSOA Code:",
            "Predicted Crime:",
            "Population:",
            "Latest Housing Price (£):"
        ],
        localize=True
    )
).add_to(m)

# -----------------------------
# Show map
# -----------------------------
m

In [ ]:
print(gdf_demo[['code', 'Predicted_Crime']].head())

        code  Predicted_Crime
0  E01000037             8.89
1  E01033729            14.98
2  E01000038             5.19
3  E01033730            -0.08
4  E01000039             6.10
